### FASE 4 da Libertadores - Semifinal

In [1]:
import cartolafc
import pandas as pd
from difflib import get_close_matches
import json
import os

pd.set_option('display.max_columns', 50)            # permite a visualização de 50 colunas do dataframe
pd.options.display.float_format = '{:.2f}'.format   # pandas: para todos os números aparecerem com duas casas decimais

# Cria uma instância da API
api = cartolafc.Api(attempts=5)

2026-05-11 09:56:32,643 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


### Carregar o arquivo Excel com os classificados e Gerar um Dicionário

In [2]:
CAMINHO_ARQUIVO = "classificados_fase_3.xlsx"

# Inicialização segura
nomes_por_id = {}
dados_torneio_semi = []

if not os.path.exists(CAMINHO_ARQUIVO):
    print("📁 Arquivo ainda não existe — aguardando fim da fase anterior.")
else:
    try:
        # Carrega o arquivo
        df_classificados = pd.read_excel(CAMINHO_ARQUIVO)

        if df_classificados.empty:
            print("📄 Arquivo existe, mas ainda está vazio. Rodadas da fase anterior não concluídas.")
        else:
            # Filtrar apenas classificados válidos
            df_validos = df_classificados[df_classificados["classificado_id"].notnull()].reset_index(drop=True)

            if len(df_validos) < 4:
                print(f"⚠️ Apenas {len(df_validos)} classificados encontrados. Aguardando rodada restante.")
            else:
                # Gerar dicionário nome → ID
                nomes_por_id = dict(zip(df_validos["classificado_id"], df_validos["classificado_nome"]))
                display(nomes_por_id)
                print("\n" + "-" * 100 + "\n")

                # Confrontos da semifinal com ordem V1 x V3 e V2 x V4
                dados_torneio_semi = [
                    ("Jogo 1 (JG1)", df_validos.loc[0, "classificado_id"]),  # V1
                    ("Jogo 1 (JG1)", df_validos.loc[2, "classificado_id"]),  # V3
                    ("Jogo 2 (JG2)", df_validos.loc[1, "classificado_id"]),  # V2
                    ("Jogo 2 (JG2)", df_validos.loc[3, "classificado_id"]),  # V4
                ]

                print("✅ Confrontos da semifinal:")
                for linha in dados_torneio_semi:
                    print(linha)

    except Exception as e:
        print(f"❌ Erro ao processar o arquivo: {e}")


📁 Arquivo ainda não existe — aguardando fim da fase anterior.


In [3]:
# Criar DataFrame base
dados_torneio_semi = pd.DataFrame(dados_torneio_semi, columns=["Jogo", "ID do Time"])

# Adicionar Nome do Time usando o dicionário
dados_torneio_semi["Nome do Time"] = dados_torneio_semi["ID do Time"].map(nomes_por_id)

# Adicionar ID no Grupo
dados_torneio_semi["ID no Grupo"] = dados_torneio_semi.groupby("Jogo").cumcount() + 1
dados_torneio_semi["ID no Grupo"] = dados_torneio_semi["ID no Grupo"].astype(str) + "_" + dados_torneio_semi["Jogo"].str[-2]

# Reorganizar colunas
df_torneio_semi_liberta = dados_torneio_semi[["Jogo", "ID do Time", "Nome do Time", "ID no Grupo"]]

df_liberta_jogo_1 = df_torneio_semi_liberta[df_torneio_semi_liberta["Jogo"] == "Jogo 1 (JG1)"]
df_liberta_jogo_2 = df_torneio_semi_liberta[df_torneio_semi_liberta["Jogo"] == "Jogo 2 (JG2)"]

# Lista de grupos
grupos = {
    "Jogo 1 (JG1)": df_liberta_jogo_1,
    "Jogo 2 (JG2)": df_liberta_jogo_2
}

display(df_liberta_jogo_1, df_liberta_jogo_2)

,Jogo,ID do Time,Nome do Time,ID no Grupo


,Jogo,ID do Time,Nome do Time,ID no Grupo


### Definição dos Confrontos das 2 Rodadas da Fase 4 da Libertadores (Rodada 15 A 16)

In [4]:
# Rodada 1 - Fase 4 Libertadores (Equivalente a 34º Rodada do Campeonato Brasileiro) 
confrontos_1a_rodada = [
    # Jogo 1 (JG1)
    ("Jogo 1 (JG1)", "1_1", "2_1"),    

    # Jogo 2 (JG2)
    ("Jogo 2 (JG2)", "1_2", "2_2")   
]

# Rodada 2 - Fase 4 Libertadores (Equivalente a 35º Rodada do Campeonato Brasileiro)
confrontos_2a_rodada = [
    # Jogo 1 (JG1)
    ("Jogo 1 (JG1)", "2_1", "1_1"),
    
    # Jogo 2 (JG2)
    ("Jogo 2 (JG2)", "2_2", "1_2") 
]

In [5]:
# Transformar em DataFrame
df_confrontos = pd.DataFrame(confrontos_1a_rodada, columns=["Grupo", "Mandante_ID", "Visitante_ID"])

# Junta com df_torneio para buscar dados dos mandantes
df_mandantes = df_torneio_semi_liberta.rename(columns={
    "ID no Grupo": "Mandante_ID",
    "Nome do Time": "Mandante_Nome",
    "ID do Time": "Mandante_ID_Time"
})[["Jogo", "Mandante_ID", "Mandante_Nome", "Mandante_ID_Time"]]

# Junta com df_torneio para buscar dados dos visitantes
df_visitantes = df_torneio_semi_liberta.rename(columns={
    "ID no Grupo": "Visitante_ID",
    "Nome do Time": "Visitante_Nome",
    "ID do Time": "Visitante_ID_Time"    
})[["Jogo", "Visitante_ID", "Visitante_Nome", "Visitante_ID_Time"]]

display(df_confrontos)

,Grupo,Mandante_ID,Visitante_ID
0,Jogo 1 (JG1),1_1,2_1
1,Jogo 2 (JG2),1_2,2_2


In [6]:
# Transformar em DataFrame
df_confrontos = pd.DataFrame(confrontos_1a_rodada, columns=["Jogo", "Mandante_ID", "Visitante_ID"])
df_confrontos["Rodada"] = 15
df_rodada_15 = df_confrontos.merge(df_mandantes, on=["Jogo", "Mandante_ID"])
df_rodada_15 = df_rodada_15.merge(df_visitantes, on=["Jogo", "Visitante_ID"])

# Transformar em DataFrame
df_confrontos_2 = pd.DataFrame(confrontos_2a_rodada, columns=["Jogo", "Mandante_ID", "Visitante_ID"])
df_confrontos_2["Rodada"] = 16
df_rodada_16 = df_confrontos_2.merge(df_mandantes, on=["Jogo", "Mandante_ID"])
df_rodada_16 = df_rodada_16.merge(df_visitantes, on=["Jogo", "Visitante_ID"])

display(df_rodada_15)

,Jogo,Mandante_ID,Visitante_ID,Rodada,Mandante_Nome,Mandante_ID_Time,Visitante_Nome,Visitante_ID_Time


In [7]:
df_rodadas = pd.concat([
    df_rodada_15,
    df_rodada_16
], ignore_index=True)

# Ajustar a numeração da rodada para refletir as rodadas do Cartola (Rodada 15 até 16)
df_rodadas["Rodada"] = df_rodadas["Rodada"]

df_rodadas.to_excel("confrontos_fase_4_libertadores.xlsx", index=False)

# Exibir os confrontos da fase 4 (Semi Finais)
display(df_rodadas.head(8)) 

,Jogo,Mandante_ID,Visitante_ID,Rodada,Mandante_Nome,Mandante_ID_Time,Visitante_Nome,Visitante_ID_Time


In [8]:
# Criar lista de dicionários no formato desejado
confrontos_js_fase_4 = []

for _, row in df_rodadas.iterrows():
    confronto = {
        "jogo": row["Jogo"],
        "rodada": int(row["Rodada"]),
        "mandante": {
            "id": int(row["Mandante_ID_Time"]),
            "nome": row["Mandante_Nome"]
        },
        "visitante": {
            "id": int(row["Visitante_ID_Time"]),
            "nome": row["Visitante_Nome"]        }
    }
    confrontos_js_fase_4.append(confronto)

# Converter para JSON formatado
json_str = json.dumps(confrontos_js_fase_4, indent=2, ensure_ascii=False)

# Salvar como arquivo JS com uma variável global
with open("confrontos_fase4_libertadores.js", "w", encoding="utf-8") as f:
    f.write("const confrontosFase4 = ")
    f.write(json_str)
    f.write(";")

In [9]:
def exibir_confrontos(df_rodadas, rodada=None, jogo=None):
    """
    Filtra e exibe os confrontos por rodada e/ou jogo.
    
    Parâmetros:
    - df_rodadas: DataFrame com todos os confrontos
    - rodada: número da rodada (int ou None para todas)
    - jogo: nome do jogo (str ou None para todos)
    
    Retorna:
    - DataFrame filtrado com as colunas relevantes
    """
    colunas = ["Rodada", "Jogo", "Mandante_Nome", "Visitante_Nome"]
    df_filtrado = df_rodadas.copy()

    df_filtrado["Rodada"] = df_filtrado["Rodada"].astype(str) + "ª Rodada"    

    if rodada is not None:
        df_filtrado = df_filtrado[df_filtrado["Rodada"] == rodada]

    if jogo is not None:
        df_filtrado = df_filtrado[df_filtrado["Jogo"] == jogo]

    return df_filtrado[colunas].sort_values(by=["Jogo", "Rodada"])

In [10]:
jogo = 2

# Exibir todos os confrontos do Jogo 2
display(exibir_confrontos(df_rodadas, jogo= f"Jogo {jogo} (JG{jogo})").head(8))

,Rodada,Jogo,Mandante_Nome,Visitante_Nome


In [11]:
# def campeonato_comecou(api, ids_times):
#     """Verifica se o campeonato já começou observando a pontuação na 1ª rodada."""
#     for time_id in ids_times.values():
#         try:
#             pontuacao = api.time(time_id=time_id, rodada=34).ultima_pontuacao
#             if pontuacao is not None:
#                 return True
#         except cartolafc.errors.CartolaFCError:
#             continue
#     return False

# def obter_pontuacao_por_rodada(api, time_id, rodada_atual):
#     """Obtém a pontuação do time em cada rodada até a rodada atual."""
#     pontuacoes = {}
#     for rodada in range(15, rodada_atual):
#         try:
#             time_rodada = api.time(time_id=time_id, rodada=rodada)
#             pontuacoes[rodada] = time_rodada.ultima_pontuacao
#         except cartolafc.errors.CartolaFCError as e:
#             print(f"Erro ao acessar pontuação da rodada {rodada} para o time {time_id}: {e}")
#             pontuacoes[rodada] = None
#     return pontuacoes


# # def gerar_df_pontuacoes(api, ids_times):
# #     rodada_atual = api.mercado().rodada_atual
# #     total_rodadas = 2    

# #     if not campeonato_comecou(api, ids_times):
# #         print("📌 O campeonato ainda não começou. Criando estrutura com placeholders.")
# #         df = pd.DataFrame(index=ids_times.keys(), columns=[f'Rodada {i}' for i in range(15, total_rodadas + 1)])
# #         df[:] = 0
# #     else:
# #         df = pd.DataFrame()
# #         for nome, time_id in ids_times.items():
# #             pontuacoes = obter_pontuacao_por_rodada(api, time_id, rodada_atual)
# #             df[nome] = pd.Series(pontuacoes)
# #         df = df.transpose()
# #         df.columns = [f'Rodada {i}' for i in range(15, rodada_atual)]
# #         df.loc['Lider_Rodada'] = df.idxmax()
    
# #     return df

# def gerar_df_pontuacoes(api, ids_times, rodada_inicio=34):
#     rodada_atual = api.mercado().rodada_atual

#     if not campeonato_comecou(api, ids_times):
#         print("📌 A fase ainda não começou. Criando estrutura com placeholders.")
#         df = pd.DataFrame(index=ids_times.keys(), columns=[f'Rodada {i}' for i in range(rodada_inicio, rodada_inicio + 2)])
#         df[:] = None  # ou 0 se preferir
#         return df

#     # Campeonato começou
#     df = pd.DataFrame()
#     for nome, time_id in ids_times.items():
#         pontuacoes = obter_pontuacao_por_rodada(api, time_id, rodada_atual)
#         df[nome] = pd.Series(pontuacoes)

#     df = df.transpose()

#     # Ajustar nome das colunas para "Rodada X"
#     colunas_numeradas = df.columns.tolist()
#     df.columns = [f'Rodada {i}' for i in colunas_numeradas]

#     # Adicionar a linha com o time líder por rodada (opcional)
#     if not df.empty:
#         df.loc['Lider_Rodada'] = df.idxmax()

#     return df


In [12]:
import requests
import time
import sys
from pathlib import Path

FASE_INICIO = 15
FASE_LIMITE = 16
PER_REQ_SLEEP = 1.0

ROOT = Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.parciais as parciais
parciais.HEADERS = globals().get("HEADERS", {})
fetch_pontuados = parciais.fetch_pontuados
fetch_time_payload = parciais.fetch_time_payload
clubes_que_ja_jogaram = parciais.clubes_que_ja_jogaram
calcular_parcial_time = parciais.calcular_parcial_time

sess = requests.Session()
sess.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})


def http_status_e_rodada():
    r = sess.get("https://api.cartola.globo.com/mercado/status", timeout=20)
    r.raise_for_status()
    data = r.json()
    return int(data.get("status_mercado", 0)), int(data.get("rodada_atual", 0))


def _coerce_float(value):
    try:
        if value is None:
            return None
        return float(value)
    except Exception:
        return None


def extrair_pontuacao_payload(payload):
    if not isinstance(payload, dict):
        return None

    candidatos = []
    for chave in ("pontos", "pontuacao"):
        candidatos.append(payload.get(chave))

    time_info = payload.get("time", {})
    if isinstance(time_info, dict):
        for chave in ("pontos", "pontuacao"):
            candidatos.append(time_info.get(chave))

    pontos_info = payload.get("pontos")
    if isinstance(pontos_info, dict):
        for chave in ("rodada", "total", "valor"):
            candidatos.append(pontos_info.get(chave))

    for candidato in candidatos:
        valor = _coerce_float(candidato)
        if valor is not None:
            return valor

    return None


def obter_pontuacao_rodada_fechada(api, time_id, rodada):
    payload = fetch_time_payload(int(time_id), int(rodada))
    pontuacao_payload = extrair_pontuacao_payload(payload)
    if pontuacao_payload is not None:
        return pontuacao_payload

    time_rodada = api.time(time_id=int(time_id), rodada=int(rodada))
    for atributo in ("pontos", "pontuacao", "ultima_pontuacao"):
        valor = _coerce_float(getattr(time_rodada, atributo, None))
        if valor is not None:
            return valor

    return None


def gerar_df_pontuacoes(api, ids_times):
    global status_http, rodada_http, rodada_api, rod_ref, FASE_FIM, RODADAS_CONCLUIDAS_FIM

    colunas_fase = [f"Rodada {i}" for i in range(FASE_INICIO, FASE_LIMITE + 1)]

    try:
        status_http, rodada_http = http_status_e_rodada()
    except Exception:
        status_http, rodada_http = 0, 0

    try:
        rodada_api = int(api.mercado().rodada_atual)
    except Exception:
        rodada_api = int(rodada_http or 0)

    rod_ref = int(rodada_http or rodada_api or 0)

    if status_http == 2:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, rod_ref))
        RODADAS_CONCLUIDAS_FIM = FASE_FIM - 1
    elif status_http == 1:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, max(rod_ref - 1, FASE_INICIO)))
        RODADAS_CONCLUIDAS_FIM = FASE_FIM
    else:
        FASE_FIM = max(FASE_INICIO, min(FASE_LIMITE, rod_ref if rod_ref else FASE_INICIO))
        RODADAS_CONCLUIDAS_FIM = FASE_FIM

    print(
        f"Status={status_http} | rodada_http={rodada_http} | rodada_api={rodada_api} | "
        f"FASE_INICIO={FASE_INICIO} | FASE_FIM={FASE_FIM} | fechadas ate {RODADAS_CONCLUIDAS_FIM}"
    )

    dados = {}
    for nome, time_id in ids_times.items():
        pontuacoes = {}
        if RODADAS_CONCLUIDAS_FIM >= FASE_INICIO:
            for rodada in range(FASE_INICIO, RODADAS_CONCLUIDAS_FIM + 1):
                try:
                    pontuacoes[rodada] = obter_pontuacao_rodada_fechada(api, time_id, rodada)
                except Exception as e:
                    print(f"Erro ao acessar pontuacao da rodada {rodada} para o time {time_id}: {e}")
                    pontuacoes[rodada] = None
        dados[nome] = {f"Rodada {rodada}": pontuacoes.get(rodada) for rodada in range(FASE_INICIO, FASE_LIMITE + 1)}

    df = pd.DataFrame.from_dict(dados, orient="index")
    df = df.reindex(columns=colunas_fase)
    df = df.apply(pd.to_numeric, errors="coerce")

    col_atual = f"Rodada {rod_ref}"
    if status_http == 2 and FASE_INICIO <= rod_ref <= FASE_LIMITE:
        print(f"Rodada {rod_ref} em andamento: aplicando parciais")
        mapa_pontuados = fetch_pontuados()
        if mapa_pontuados:
            clubes_jogaram = clubes_que_ja_jogaram(rod_ref)
            for nome_time, time_id in ids_times.items():
                try:
                    total = calcular_parcial_time(int(time_id), rod_ref, mapa_pontuados, clubes_jogaram)
                    if col_atual not in df.columns:
                        df[col_atual] = pd.NA
                    df.loc[nome_time, col_atual] = round(total, 2)
                except Exception as e:
                    print(f"Erro ao calcular parcial da rodada {rod_ref} para o time {time_id}: {e}")
                time.sleep(PER_REQ_SLEEP)
        else:
            print("Parciais indisponiveis no momento.")
    else:
        print("Sem parciais para aplicar nesta fase.")

    colunas_com_dados = [col for col in df.columns if df[col].notna().any()]
    if colunas_com_dados:
        df.loc["Lider_Rodada", colunas_com_dados] = df[colunas_com_dados].idxmax()
    if not colunas_com_dados:
        df.loc["Lider_Rodada"] = None

    return df


In [13]:
ids_times = {v: k for k, v in nomes_por_id.items()}
df_pontuacoes = gerar_df_pontuacoes(api, ids_times)

if df_pontuacoes.empty:
    print("⚠️ Sem pontuações disponíveis ainda.")
else:
    display(df_pontuacoes.T)


Status=2 | rodada_http=15 | rodada_api=15 | FASE_INICIO=15 | FASE_FIM=15 | fechadas ate 14
Rodada 15 em andamento: aplicando parciais


,Lider_Rodada
Rodada 15,NaN
Rodada 16,NaN


In [14]:
# ids_times = {v: k for k, v in nomes_por_id.items()}

# df_pontuacoes = gerar_df_pontuacoes(api, ids_times)
# display(df_pontuacoes.T)

In [15]:
# def classificacao_por_grupo(df_rodadas, df_pontuacoes):

#     df_pontuacoes_times = df_pontuacoes.drop(index='Lider_Rodada', errors='ignore')
#     estatisticas = {}

#     for _, confronto in df_rodadas.iterrows():
#         rodada = confronto["Rodada"]
#         mandante = confronto["Mandante_Nome"]
#         visitante = confronto["Visitante_Nome"]
#         jogo = confronto["Jogo"]
#         coluna_rodada = f"Rodada {rodada}"

#         if mandante not in df_pontuacoes_times.index or visitante not in df_pontuacoes_times.index:
#             continue
#         if coluna_rodada not in df_pontuacoes_times.columns:
#             continue

#         pontos_mandante = df_pontuacoes_times.at[mandante, coluna_rodada]
#         pontos_visitante = df_pontuacoes_times.at[visitante, coluna_rodada]

#         # Ignorar confrontos ainda não disputados (com pontuação 0 ou ausente)
#         if (
#             pd.isnull(pontos_mandante) or pd.isnull(pontos_visitante) or
#             (pontos_mandante == 0 and pontos_visitante == 0)
#         ):
#             continue

#         for time in [mandante, visitante]:
#             if jogo not in estatisticas:
#                 estatisticas[jogo] = {}
#             if time not in estatisticas[jogo]:
#                 estatisticas[jogo][time] = {
#                     "Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
#                     "Total_Cartola": 0,
#                     "Cartola_Sofrido": 0
#                 }

#         # Atualizar estatísticas do jogo
#         estatisticas[jogo][mandante]["Total_Cartola"] += pontos_mandante
#         estatisticas[jogo][mandante]["Cartola_Sofrido"] += pontos_visitante

#         estatisticas[jogo][visitante]["Total_Cartola"] += pontos_visitante
#         estatisticas[jogo][visitante]["Cartola_Sofrido"] += pontos_mandante

#         if pontos_mandante > pontos_visitante:
#             estatisticas[jogo][mandante]["Pontos"] += 3
#             estatisticas[jogo][mandante]["Vitórias"] += 1
#             estatisticas[jogo][visitante]["Derrotas"] += 1
#         elif pontos_mandante < pontos_visitante:
#             estatisticas[jogo][visitante]["Pontos"] += 3
#             estatisticas[jogo][visitante]["Vitórias"] += 1
#             estatisticas[jogo][mandante]["Derrotas"] += 1
#         else:
#             estatisticas[jogo][mandante]["Pontos"] += 1
#             estatisticas[jogo][visitante]["Pontos"] += 1
#             estatisticas[jogo][mandante]["Empates"] += 1
#             estatisticas[jogo][visitante]["Empates"] += 1

#     # Gerar DataFrame final
#     df_resultado = pd.concat([
#         pd.DataFrame({
#             "Jogo": jogo,
#             "Nome do Time": list(times.keys()),
#             "Pontos": [stats["Pontos"] for stats in times.values()],
#             "Vitórias": [stats["Vitórias"] for stats in times.values()],
#             "Empates": [stats["Empates"] for stats in times.values()],
#             "Derrotas": [stats["Derrotas"] for stats in times.values()],
#             "Total Cartola": [stats["Total_Cartola"] for stats in times.values()],
#             "Cartola Sofrido": [stats["Cartola_Sofrido"] for stats in times.values()],
#             "Saldo Cartola": [
#                 stats["Total_Cartola"] - stats["Cartola_Sofrido"] for stats in times.values()
#             ]
#         }) for jogo, times in estatisticas.items()
#     ], ignore_index=True)

#     # Ordenar e adicionar posição
#     df_resultado = df_resultado.sort_values(
#         by=["Jogo", "Pontos", "Vitórias", "Total Cartola", "Saldo Cartola"],
#         ascending=[True, False, False, False, False]
#     )

#     df_resultado["Posição"] = df_resultado.groupby("Jogo").cumcount() + 1

#     df_resultado_por_jogo = {
#         jogo: df_resultado[df_resultado["Jogo"] == jogo] for jogo in df_resultado["Jogo"].unique()
#     }

#     return df_resultado, df_resultado_por_jogo


In [16]:
def classificacao_por_grupo(df_rodadas, df_pontuacoes, rodada_atual=None):
    """
    Gera classificação por grupo (ou jogo), exibindo times zerados
    se a fase ainda não começou (sem pontuações nas rodadas da fase).
    """
    df_pontuacoes_times = df_pontuacoes.drop(index='Lider_Rodada', errors='ignore')
    estatisticas = {}

    # 🔹 Descobre rodadas da fase
    rodadas_fase = sorted(df_rodadas["Rodada"].unique())
    rodadas_validas = [r for r in rodadas_fase if f"Rodada {r}" in df_pontuacoes.columns]

    # 🔹 Se rodada_atual for anterior à primeira rodada da fase → retorna tabela zerada
    if rodada_atual is not None and rodada_atual < min(rodadas_fase):
        print(f"📅 Fase ainda não começou (rodada atual {rodada_atual}). Exibindo zerado.")
        return montar_tabela_zerada(df_rodadas)

    # 🔹 Verifica se há alguma pontuação válida
    tem_pontuacao = any(
        pd.notna(df_pontuacoes[f"Rodada {r}"]).any() for r in rodadas_validas
    )
    if not tem_pontuacao:
        print("📊 Nenhuma pontuação registrada nesta fase. Exibindo classificação inicial zerada.")
        return montar_tabela_zerada(df_rodadas)

    # 🔹 Caso existam pontuações, gera normalmente
    for _, confronto in df_rodadas.iterrows():
        rodada = confronto.get("Rodada")
        mandante = confronto.get("Mandante_Nome")
        visitante = confronto.get("Visitante_Nome")
        jogo = confronto.get("Jogo")
        coluna_rodada = f"Rodada {rodada}"

        if coluna_rodada not in df_pontuacoes_times.columns:
            continue

        pontos_mandante = df_pontuacoes_times.at[mandante, coluna_rodada]
        pontos_visitante = df_pontuacoes_times.at[visitante, coluna_rodada]
        pontos_mandante = 0 if pd.isnull(pontos_mandante) else pontos_mandante
        pontos_visitante = 0 if pd.isnull(pontos_visitante) else pontos_visitante

        if jogo not in estatisticas:
            estatisticas[jogo] = {}
        for time in [mandante, visitante]:
            estatisticas[jogo].setdefault(time, {
                "Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                "Total_Cartola": 0, "Cartola_Sofrido": 0
            })

        estatisticas[jogo][mandante]["Total_Cartola"] += pontos_mandante
        estatisticas[jogo][mandante]["Cartola_Sofrido"] += pontos_visitante
        estatisticas[jogo][visitante]["Total_Cartola"] += pontos_visitante
        estatisticas[jogo][visitante]["Cartola_Sofrido"] += pontos_mandante

        # Pula se ainda não houve pontuação real
        if pontos_mandante == 0 and pontos_visitante == 0:
            continue

        if pontos_mandante > pontos_visitante:
            estatisticas[jogo][mandante]["Pontos"] += 3
            estatisticas[jogo][mandante]["Vitórias"] += 1
            estatisticas[jogo][visitante]["Derrotas"] += 1
        elif pontos_mandante < pontos_visitante:
            estatisticas[jogo][visitante]["Pontos"] += 3
            estatisticas[jogo][visitante]["Vitórias"] += 1
            estatisticas[jogo][mandante]["Derrotas"] += 1
        else:
            estatisticas[jogo][mandante]["Pontos"] += 1
            estatisticas[jogo][visitante]["Pontos"] += 1
            estatisticas[jogo][mandante]["Empates"] += 1
            estatisticas[jogo][visitante]["Empates"] += 1

    # 🔹 Monta dataframe final
    df_resultado = pd.concat([
        pd.DataFrame({
            "Jogo": jogo,
            "Nome do Time": list(times.keys()),
            "Pontos": [stats["Pontos"] for stats in times.values()],
            "Vitórias": [stats["Vitórias"] for stats in times.values()],
            "Empates": [stats["Empates"] for stats in times.values()],
            "Derrotas": [stats["Derrotas"] for stats in times.values()],
            "Total Cartola": [stats["Total_Cartola"] for stats in times.values()],
            "Cartola Sofrido": [stats["Cartola_Sofrido"] for stats in times.values()],
            "Saldo Cartola": [
                stats["Total_Cartola"] - stats["Cartola_Sofrido"] for stats in times.values()
            ]
        })
        for jogo, times in estatisticas.items()
    ], ignore_index=True)

    df_resultado = df_resultado.sort_values(
        by=["Jogo", "Pontos", "Vitórias", "Total Cartola", "Saldo Cartola"],
        ascending=[True, False, False, False, False]
    )
    df_resultado["Posição"] = df_resultado.groupby("Jogo").cumcount() + 1

    df_resultado_por_jogo = {
        jogo: df_resultado[df_resultado["Jogo"] == jogo] for jogo in df_resultado["Jogo"].unique()
    }

    return df_resultado, df_resultado_por_jogo


# 🔧 Função auxiliar
def montar_tabela_zerada(df_rodadas):
    """Cria tabela zerada com base nos confrontos informados."""
    estatisticas = {
        jogo: {
            row["Mandante_Nome"]: {"Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                                   "Total_Cartola": 0, "Cartola_Sofrido": 0},
            row["Visitante_Nome"]: {"Pontos": 0, "Vitórias": 0, "Empates": 0, "Derrotas": 0,
                                    "Total_Cartola": 0, "Cartola_Sofrido": 0}
        }
        for jogo, row in df_rodadas.groupby("Jogo").first().iterrows()
    }

    df_resultado = pd.concat([
        pd.DataFrame({
            "Jogo": jogo,
            "Nome do Time": list(times.keys()),
            "Pontos": [0, 0],
            "Vitórias": [0, 0],
            "Empates": [0, 0],
            "Derrotas": [0, 0],
            "Total Cartola": [0, 0],
            "Cartola Sofrido": [0, 0],
            "Saldo Cartola": [0, 0],
        })
        for jogo, times in estatisticas.items()
    ], ignore_index=True)

    df_resultado["Posição"] = 0
    df_resultado_por_jogo = {
        jogo: df_resultado[df_resultado["Jogo"] == jogo] for jogo in df_resultado["Jogo"].unique()
    }
    return df_resultado, df_resultado_por_jogo


In [17]:
# def obter_classificados_e_perdedores(resultados, ids_por_nome):
#     """
#     Recebe resultados de ida e volta e define classificados e perdedores.
#     Critério: soma dos pontos (ida + volta). Em caso de empate, decide por
#     maior total Cartola acumulado; se persistir, empate declarado.
#     """
#     classificados, perdedores = [], []
#     jogos_agrupados = {}

#     for jogo in resultados:
#         chave = jogo["jogo"]
#         jogos_agrupados.setdefault(chave, []).append(jogo)

#     for chave, partidas in jogos_agrupados.items():
#         if len(partidas) < 2:
#             continue

#         partidas = sorted(partidas, key=lambda x: x["rodada"])
#         ida, volta = partidas

#         if ida["mandante"]["pontos"] is None or volta["mandante"]["pontos"] is None:
#             continue

#         time1 = ida["mandante"]["nome"]
#         time2 = ida["visitante"]["nome"]

#         # Soma de pontos Cartola ida+volta
#         pontos_time1 = ida["mandante"]["pontos"] + volta["visitante"]["pontos"]
#         pontos_time2 = ida["visitante"]["pontos"] + volta["mandante"]["pontos"]

#         # Critério primário: maior soma de pontos
#         if pontos_time1 > pontos_time2:
#             vencedor, perdedor = time1, time2
#         elif pontos_time2 > pontos_time1:
#             vencedor, perdedor = time2, time1
#         else:
#             # Critério de desempate: maior total acumulado no confronto
#             total1 = ida["mandante"]["pontos"] + ida["visitante"]["pontos"] + volta["mandante"]["pontos"] + volta["visitante"]["pontos"]
#             total2 = total1  # mesma soma, mas pode ajustar conforme metadados externos

#             if total1 > total2:
#                 vencedor, perdedor = time1, time2
#             elif total2 > total1:
#                 vencedor, perdedor = time2, time1
#             else:
#                 vencedor, perdedor = "EMPATE", "EMPATE"

#         classificados.append({
#             "jogo": chave,
#             "classificado_nome": vencedor,
#             "classificado_id": ids_por_nome.get(vencedor) if vencedor in ids_por_nome else None
#         })
#         perdedores.append({
#             "jogo": chave,
#             "perdedor_nome": perdedor,
#             "perdedor_id": ids_por_nome.get(perdedor) if perdedor in ids_por_nome else None
#         })

#     return classificados, perdedores


In [18]:
# # Padroniza os nomes no df_rodadas
# df_rodadas["Mandante_Nome"] = df_rodadas["Mandante_Nome"].str.strip()
# df_rodadas["Visitante_Nome"] = df_rodadas["Visitante_Nome"].str.strip()

# # Padroniza os índices do df_pontuacoes
# df_pontuacoes.index = df_pontuacoes.index.str.strip()

# # Exibe para conferência
# display(df_pontuacoes)

In [ ]:
# Padroniza e exibe as pontuações se existirem
try:
    if 'df_pontuacoes' not in globals() or df_pontuacoes.empty:
        print("📌 Nenhuma pontuação disponível — provavelmente ainda não há classificados definidos.")
    else:
        # Padroniza os nomes no df_rodadas
        df_rodadas["Mandante_Nome"] = df_rodadas["Mandante_Nome"].str.strip()
        df_rodadas["Visitante_Nome"] = df_rodadas["Visitante_Nome"].str.strip()

        # Padroniza os índices do df_pontuacoes
        df_pontuacoes.index = df_pontuacoes.index.str.strip()

        # Exibe
        display(df_pontuacoes)

except NameError as e:
    print("❌ Variável df_rodadas ou df_pontuacoes não está definida ainda.")
except Exception as e:
    print(f"❌ Erro inesperado: {e}")


In [ ]:
# # Gerar a classificação da fase 4
# df_resultado, df_resultado_por_jogo = classificacao_por_grupo(
#     df_rodadas, df_pontuacoes
# )

# # Salvar cada grupo em uma aba do Excel
# with pd.ExcelWriter("classificacao_por_grupo_fase_4.xlsx") as writer:
#     for jogo, df in df_resultado_por_jogo.items():
#         df.to_excel(writer, sheet_name=jogo, index=False)

# # Exibir a classificação geral
# df_resultado_jogo_1 = df_resultado[df_resultado["Jogo"] == "Jogo 1 (JG1)"]
# df_resultado_jogo_2 = df_resultado[df_resultado["Jogo"] == "Jogo 2 (JG2)"]

# display(df_resultado_jogo_1, df_resultado_jogo_2)

In [ ]:
try:
    # Verifica se as variáveis existem e estão válidas
    if (
        'df_rodadas' not in globals() or df_rodadas.empty or
        'df_pontuacoes' not in globals() or df_pontuacoes.empty
    ):
        print("⚠️ Rodadas ou pontuações ainda não disponíveis — classificação não será gerada.")
    
    elif 'classificacao_por_grupo' not in globals():
        print("⚠️ Função 'classificacao_por_grupo' não está definida.")
    
    else:
        # Gerar a classificação da fase 4
        df_resultado, df_resultado_por_jogo = classificacao_por_grupo(
            df_rodadas, df_pontuacoes
        )

        # Salvar cada grupo em uma aba do Excel
        with pd.ExcelWriter("classificacao_por_grupo_fase_4.xlsx") as writer:
            for jogo, df in df_resultado_por_jogo.items():
                df.to_excel(writer, sheet_name=jogo, index=False)

        # Exibir a classificação geral
        df_resultado_jogo_1 = df_resultado[df_resultado["Jogo"] == "Jogo 1 (JG1)"]
        df_resultado_jogo_2 = df_resultado[df_resultado["Jogo"] == "Jogo 2 (JG2)"]

        display(df_resultado_jogo_1, df_resultado_jogo_2)

except Exception as e:
    print(f"❌ Erro ao gerar a classificação: {e}")


In [ ]:
# Verifica se a variável df_resultado_por_grupo existe e está populada
if 'df_resultado_por_jogo' in locals() and df_resultado_por_jogo:
    # Criar estrutura em formato de dicionário para JSON/JS
    classificacao_js_fase_4 = {}

    for jogo, df in df_resultado_por_jogo.items():
        classificacao_js_fase_4[jogo] = []

        # Remove duplicatas
        df = df.drop_duplicates(subset=["Nome do Time"])

        for _, row in df.iterrows():
            classificacao_js_fase_4[jogo].append({
                "posicao": int(row["Posição"]),
                "nome": row["Nome do Time"],
                "pontos": int(row["Pontos"]),
                "vitorias": int(row["Vitórias"]),
                "empates": int(row["Empates"]),
                "derrotas": int(row["Derrotas"]),
                "totalCartola": float(row["Total Cartola"]),
                "cartolaSofrido": float(row["Cartola Sofrido"]),
                "saldoCartola": float(row["Saldo Cartola"])
            })


    # Converter para JSON formatado
    json_str = json.dumps(classificacao_js_fase_4, indent=2, ensure_ascii=False)

    # Salvar como arquivo JS com uma variável global
    with open("classificacao_por_grupo_fase_4.js", "w", encoding="utf-8") as f:
        f.write("const classificacaoFase4 = ")
        f.write(json_str)
        f.write(";")

    print("✅ Arquivo JS salvo com sucesso.")

else:
    print("⚠️ Classificação da fase 4 indisponível. Criando placeholders.")

    # ⚠️ Classificação da fase 4 indisponível — criar estrutura zerada manualmente

    # Dicionário com confrontos definidos
    confrontos_fase_4 = {
        "Jogo 1 (JG1)": ["Analove10 ITAQUI GRANDE!!", "Real SCI"],
        "Jogo 2 (JG2)": ["Texas Club 2025", "Lá do Itaqui"]
    }

    # Estrutura zerada para o JS
    classificacao_js_fase_4 = {}

    for jogo, times in confrontos_fase_4.items():
        classificacao_js_fase_4[jogo] = []
        for nome_time in times:
            classificacao_js_fase_4[jogo].append({
                "posicao": 0,
                "nome": nome_time,
                "pontos": 0,
                "vitorias": 0,
                "empates": 0,
                "derrotas": 0,
                "totalCartola": 0.0,
                "cartolaSofrido": 0.0,
                "saldoCartola": 0.0
            })

    # Salvar como JS
    with open("classificacao_por_grupo_fase_4.js", "w", encoding="utf-8") as f:
        f.write("const classificacaoFase4 = ")
        json.dump(classificacao_js_fase_4, f, indent=2, ensure_ascii=False)
        f.write(";")

    print("✅ Arquivo JS com classificação zerada gerado com sucesso.")


In [ ]:
def exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada, jogo=None):
    """
    Exibe os resultados de uma rodada específica, com pontuação e dados dos times.
    """

    if rodada not in df_rodadas["Rodada"].values:
        return pd.DataFrame([{
            "Jogo": jogo or "-",
            "Rodada": rodada,
            "Mandante_Nome": "-",          
            "Mandante_Pontos": "-",
            "Visitante_Nome": "-",           
            "Visitante_Pontos": "-",
        }])

    df_filtrado = df_rodadas[df_rodadas["Rodada"] == rodada]
    if jogo:
        df_filtrado = df_filtrado[df_filtrado["Jogo"] == jogo]

    resultados = []

    for _, row in df_filtrado.iterrows():
        jogo_ = row["Jogo"]
        mandante = row["Mandante_Nome"]
        visitante = row["Visitante_Nome"]

        pontos_mandante = df_pontuacoes.get(f"Rodada {rodada}", {}).get(mandante, None)
        pontos_visitante = df_pontuacoes.get(f"Rodada {rodada}", {}).get(visitante, None)

        resultados.append({
            "Jogo": jogo_,
            "Rodada": rodada,
            "Mandante_Nome": mandante,
            "Mandante_Pontos": pontos_mandante,
            "Visitante_Nome": visitante,
            "Visitante_Pontos": pontos_visitante
        })

    return pd.DataFrame(resultados)


In [ ]:
# Exibir resultados da 15ª rodada
df_resultados_rodada_15 = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=15)

# Exibir apenas os resultados do Grupo B na 1ª rodada
df_resultados_jogo_1 = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=15, jogo="Jogo 1 (JG1)")

# Exibir
display(df_resultados_rodada_15)

In [ ]:
# Criar arquivo com uma aba para cada rodada contendo os resultados detalhados
from pathlib import Path

# Caminho do arquivo de saída
caminho_resultados = "resultados_fase_4.xlsx"

# Descobrir as rodadas únicas no DataFrame
rodadas_disponiveis = sorted(df_rodadas["Rodada"].unique())

with pd.ExcelWriter(caminho_resultados) as writer:
    for rodada in rodadas_disponiveis:
        df_resultados = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=rodada)
        nome_aba = f"Rodada {rodada}"
        df_resultados.to_excel(writer, sheet_name=nome_aba, index=False)

print(f"Arquivo salvo com sucesso: {Path(caminho_resultados).resolve()}")

if "df_resultados" in locals():
    display(df_resultados)
else:
    display(pd.DataFrame())

In [ ]:
resultados_js = []

for rodada in sorted(df_rodadas["Rodada"].unique()):
    df_resultados = exibir_resultados_rodada(df_rodadas, df_pontuacoes, rodada=rodada)
    
    for _, row in df_resultados.iterrows():
        resultado = {
            "jogo": row["Jogo"],
            "rodada": int(rodada),
            "mandante": {
                "nome": row["Mandante_Nome"],
                "pontos": float(row["Mandante_Pontos"]) if row["Mandante_Pontos"] is not None else None
            },
            "visitante": {
                "nome": row["Visitante_Nome"],
                "pontos": float(row["Visitante_Pontos"]) if row["Visitante_Pontos"] is not None else None
            },
            "vencedor": (
                "mandante" if row["Mandante_Pontos"] is not None and row["Visitante_Pontos"] is not None and row["Mandante_Pontos"] > row["Visitante_Pontos"]
                else "visitante" if row["Mandante_Pontos"] is not None and row["Visitante_Pontos"] is not None and row["Mandante_Pontos"] < row["Visitante_Pontos"]
                else "empate" if row["Mandante_Pontos"] == row["Visitante_Pontos"] and row["Mandante_Pontos"] is not None
                else "indefinido"
            )

        }
        resultados_js.append(resultado)

# Exportar para arquivo .js
import json

with open("resultados_fase_4.js", "w", encoding="utf-8") as f:
    f.write("const resultadosFase4 = ")
    f.write(json.dumps(resultados_js, indent=2, ensure_ascii=False))
    f.write(";\n")

# Exporta meta de parcial (usada no front)
try:
    rodada_ref = int(rod_ref)
except Exception:
    rodada_ref = 0

parcial_payload = {"rodada": rodada_ref, "times": {}}
try:
    col_parcial = f"Rodada {rodada_ref}"
    if col_parcial in df_pontuacoes.columns:
        times_map = {}
        for nome in df_pontuacoes.index:
            if nome not in ids_times:
                continue
            try:
                val = df_pontuacoes.at[nome, col_parcial]
            except Exception:
                continue
            if str(val) in ("", "nan"):
                continue
            try:
                times_map[str(ids_times[nome])] = float(val)
            except Exception:
                continue
        parcial_payload["times"] = times_map
except Exception:
    pass

try:
    status_http_val = int(status_http)
except Exception:
    status_http_val = None

liberta_meta = {
    "rodada_atual": rodada_ref,
    "parcial_disponivel": (status_http_val == 2)
}

with open("resultados_fase_4.js", "a", encoding="utf-8") as f:
    f.write("\nconst pontuacaoParcialRodadaAtual = ")
    f.write(json.dumps(parcial_payload, indent=2, ensure_ascii=False))
    f.write(";\n")
    f.write("window.libertaMeta = ")
    f.write(json.dumps(liberta_meta, indent=2, ensure_ascii=False))
    f.write(";\n")



### Identificando Vencedores das Semifinais

In [ ]:
# def obter_classificados_com_id(resultados, ids_por_nome):
#     classificados = []
#     jogos_agrupados = {}
#     for jogo in resultados:
#         chave = jogo['jogo']
#         if chave not in jogos_agrupados:
#             jogos_agrupados[chave] = []
#         jogos_agrupados[chave].append(jogo)

#     for chave, partidas in jogos_agrupados.items():
#         if len(partidas) < 2:
#             continue

#         partidas = sorted(partidas, key=lambda x: x['rodada'])
#         ida, volta = partidas

#         if ida['mandante']['pontos'] is None or volta['mandante']['pontos'] is None:
#             continue

#         time1 = ida['mandante']['nome']
#         time2 = ida['visitante']['nome']

#         pontos_time1 = ida['mandante']['pontos'] + volta['visitante']['pontos']
#         pontos_time2 = ida['visitante']['pontos'] + volta['mandante']['pontos']

#         if pontos_time1 > pontos_time2:
#             vencedor = time1
#         elif pontos_time2 > pontos_time1:
#             vencedor = time2
#         else:
#             vencedor = "EMPATE"

#         classificados.append({
#             "jogo": chave,
#             "classificado_nome": vencedor,
#             "classificado_id": ids_por_nome.get(vencedor) if vencedor in ids_por_nome else None
#         })

#     return classificados


In [ ]:
def obter_classificados_e_perdedores(resultados, ids_por_nome):
    classificados = []
    perdedores = []
    jogos_agrupados = {}

    for jogo in resultados:
        chave = jogo['jogo']
        if chave not in jogos_agrupados:
            jogos_agrupados[chave] = []
        jogos_agrupados[chave].append(jogo)

    for chave, partidas in jogos_agrupados.items():
        if len(partidas) < 2:
            continue

        partidas = sorted(partidas, key=lambda x: x['rodada'])
        ida, volta = partidas

        if ida['mandante']['pontos'] is None or volta['mandante']['pontos'] is None:
            continue

        time1 = ida['mandante']['nome']
        time2 = ida['visitante']['nome']

        pontos_time1 = ida['mandante']['pontos'] + volta['visitante']['pontos']
        pontos_time2 = ida['visitante']['pontos'] + volta['mandante']['pontos']

        if pontos_time1 > pontos_time2:
            vencedor = time1
            perdedor = time2
        elif pontos_time2 > pontos_time1:
            vencedor = time2
            perdedor = time1
        else:
            # Se empatar, decidir por critério de desempate se desejar
            vencedor = "EMPATE"
            perdedor = "EMPATE"

        classificados.append({
            "jogo": chave,
            "classificado_nome": vencedor,
            "classificado_id": ids_por_nome.get(vencedor) if vencedor in ids_por_nome else None
        })

        perdedores.append({
            "jogo": chave,
            "perdedor_nome": perdedor,
            "perdedor_id": ids_por_nome.get(perdedor) if perdedor in ids_por_nome else None
        })

    return classificados, perdedores


In [ ]:
# Inverter os nomes
ids_por_nome = {v: k for k, v in nomes_por_id.items()}

# Ler o resultados_fase_4.js
with open("resultados_fase_4.js", "r", encoding="utf-8") as f:
    conteudo = f.read()

conteudo_json = conteudo.replace("const resultadosFase4 = ", "", 1)
conteudo_json = conteudo_json.split("\nconst pontuacaoParcialRodadaAtual = ", 1)[0].strip().rstrip(";")
resultados = json.loads(conteudo_json)

# Obter classificados e perdedores
classificados, perdedores = obter_classificados_e_perdedores(resultados, ids_por_nome)

# Salvar os classificados
df_classificados = pd.DataFrame(classificados)
df_classificados.to_excel("classificados_fase_4.xlsx", index=False)

with open("classificados_fase_4.js", "w", encoding="utf-8") as f:
    f.write("const classificadosFase4 = ")
    json.dump(classificados, f, ensure_ascii=False, indent=2)
    f.write(";")

# Salvar os perdedores
df_perdedores = pd.DataFrame(perdedores)
df_perdedores.to_excel("perdedores_fase_4.xlsx", index=False)

with open("perdedores_fase_4.js", "w", encoding="utf-8") as f:
    f.write("const perdedoresFase4 = ")
    json.dump(perdedores, f, ensure_ascii=False, indent=2)
    f.write(";")

print("✅ Classificados e perdedores salvos com sucesso em arquivos .xlsx e .js.")

# Exibir para validar
print("Classificados encontrados:", classificados)
print("Perdedores encontrados:", perdedores)



In [ ]:
# # Inverter os nomes
# ids_por_nome = {v: k for k, v in nomes_por_id.items()}

# # Ler o resultados_fase_4.js (como já fizemos antes)
# with open("resultados_fase_4.js", "r", encoding="utf-8") as f:
#     conteudo = f.read()

# conteudo_json = conteudo.replace("const resultadosFase4 = ", "").strip().rstrip(";")
# resultados = json.loads(conteudo_json)

# # Obter os classificados
# classificados = obter_classificados_com_id(resultados, ids_por_nome)

# # Salvar
# df_classificados = pd.DataFrame(classificados)
# df_classificados.to_excel("classificados_fase_4.xlsx", index=False)

# with open("classificados_fase_4.js", "w", encoding="utf-8") as f:
#     f.write("const classificadosFase4 = ")
#     json.dump(classificados, f, ensure_ascii=False, indent=2)
#     f.write(";")
# print("✅ Classificados salvos com sucesso em 'classificados_fase_4.xlsx' e 'classificados_fase_4.js'.")

# # Exibir para validar
# print("Classificados encontrados:", classificados)
